# 03 · Which learners solve Kryptonite-n? (Table 1)

**Hypothesis H2.** Learners that can *learn features* (an MLP trained by SGD, or an explicit sparse feature
selector) reach the 90% target, while learners whose representation is fixed in advance (kernels, $k$-NN)
or that need marginal signal for greedy splits (trees) stay near chance, as the theory in Section 2 predicts.

**Design.** Same 5 stratified 60/20/20 splits for every model (paired comparison).  Hyper-parameters are
chosen on the validation split only; the test split is used once per model and seed.  We report the mean
test accuracy and a 95% Student-t confidence interval over the 5 splits.

In [1]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from krypto.data import load, split, NS, SEEDS
from krypto.polynomial import poly_logreg, mock_kernel_accuracies, SparsePolyLogReg
from krypto.selection import greedy_forward_selection
from krypto.mlp import mlp, train, accuracy, set_seed, n_trainable
from krypto.stats import mean_ci, fmt, save_json
from krypto.energy import track

results = {n: {} for n in NS}
seconds = {n: {} for n in NS}          # wall-clock time of each fit (including its hyper-parameter search)


def run(n, model, fit_and_score):
    t0 = time.time()
    acc = fit_and_score()
    seconds[n].setdefault(model, []).append(time.time() - t0)
    results[n].setdefault(model, []).append(float(acc))

## 1. MLP hyper-parameters (validation split of seed 0 only)
Width 512, Adam(W) with learning rate $10^{-3}$, batch 128, early stopping on validation accuracy
(patience 30, at most 300 epochs).  Grid: depth $\in\{2,3\}$ $\times$ dropout $\in\{0, 0.2\}$.

In [2]:
GRID = [{"depth": d, "dropout": p} for d in (2, 3) for p in (0.0, 0.2)]
mlp_config, seed0_nets = {}, {}
for n in NS:
    X, y = load(f"kryptonite-{n}")
    X_tr, y_tr, X_va, y_va, X_te, y_te = split(X, y, 0)
    scores = []
    with track(f"03_mlp_grid_n{n}"):
        for cfg in GRID:
            set_seed(0)
            net, info = train(mlp(n, 512, **cfg), X_tr, y_tr, X_va, y_va, seed=0)
            scores.append((info["val_acc"], cfg, net))
            print(f"n={n} {cfg}: val {info['val_acc']:.4f} ({info['epochs']} epochs)")
    best_val, best_cfg, best_net = max(scores, key=lambda t: t[0])
    mlp_config[n], seed0_nets[n] = best_cfg, best_net
    print(f"n={n}: chosen {best_cfg} (val {best_val:.4f}); {n_trainable(best_net):,} parameters")
save_json({str(n): c for n, c in mlp_config.items()}, "mlp_config.json")

n=20 {'depth': 2, 'dropout': 0.0}: val 0.9029 (115 epochs)


n=20 {'depth': 2, 'dropout': 0.2}: val 0.9133 (124 epochs)


n=20 {'depth': 3, 'dropout': 0.0}: val 0.9100 (66 epochs)


n=20 {'depth': 3, 'dropout': 0.2}: val 0.9396 (223 epochs)
[track] 03_mlp_grid_n20: 184.1 s
n=20: chosen {'depth': 3, 'dropout': 0.2} (val 0.9396); 536,577 parameters


n=30 {'depth': 2, 'dropout': 0.0}: val 0.8760 (123 epochs)


n=30 {'depth': 2, 'dropout': 0.2}: val 0.9267 (152 epochs)


... (2 outputs left out)

n=40 {'depth': 2, 'dropout': 0.0}: val 0.9250 (58 epochs)


n=40 {'depth': 2, 'dropout': 0.2}: val 0.9537 (139 epochs)


n=40 {'depth': 3, 'dropout': 0.0}: val 0.9431 (69 epochs)


n=40 {'depth': 3, 'dropout': 0.2}: val 0.9581 (127 epochs)
[track] 03_mlp_grid_n40: 385.1 s
n=40: chosen {'depth': 3, 'dropout': 0.2} (val 0.9581); 546,817 parameters


## 2. All models on the 5 splits

In [3]:
def knn(X_tr, y_tr, X_va, y_va, X_te, y_te):
    k = max([1, 5, 15, 51, 151], key=lambda k: KNeighborsClassifier(k).fit(X_tr, y_tr).score(X_va, y_va))
    return KNeighborsClassifier(k).fit(X_tr, y_tr).score(X_te, y_te)


def mlp_run(n, seed, X_tr, y_tr, X_va, y_va, X_te, y_te):
    if seed == 0:
        return accuracy(seed0_nets[n], X_te, y_te)          # trained during the grid search above
    set_seed(seed)
    net, _ = train(mlp(n, 512, **mlp_config[n]), X_tr, y_tr, X_va, y_va, seed=seed)
    return accuracy(net, X_te, y_te)


def sparse_poly(X_tr, y_tr, X_va, y_va, X_te, y_te):
    S, _ = greedy_forward_selection(X_tr, y_tr, X_va, y_va)
    fits = [SparsePolyLogReg(S, 10, C).fit(X_tr, y_tr) for C in (0.85, 10, 100, 1000, 10000)]
    return max(fits, key=lambda m: m.score(X_va, y_va)).score(X_te, y_te)


for n in NS:
    X, y = load(f"kryptonite-{n}")
    for seed in SEEDS:
        X_tr, y_tr, X_va, y_va, X_te, y_te = D = split(X, y, seed)
        t0 = time.time()
        with track(f"03_models_n{n}_s{seed}"):
            run(n, "Logistic regression (linear)",
                lambda: LogisticRegression(C=0.85, max_iter=2000).fit(X_tr, y_tr).score(X_te, y_te))
            run(n, "Quadratic LR (mock paper)", lambda: poly_logreg(X_tr, y_tr, X_te, y_te, 2))
            run(n, "Poly-kernel LR, d=10, all bits (mock)",
                lambda: mock_kernel_accuracies(X_tr, y_tr, X_te, y_te, [10], verbose=False)[10])
            run(n, "k-nearest neighbours (k tuned)", lambda: knn(*D))
            run(n, "RBF-kernel SVM", lambda: SVC(C=10, gamma="scale").fit(X_tr, y_tr).score(X_te, y_te))
            run(n, "Random forest (300 trees)",
                lambda: RandomForestClassifier(300, n_jobs=2, random_state=seed).fit(X_tr, y_tr).score(X_te, y_te))
            run(n, "Gradient-boosted trees",
                lambda: HistGradientBoostingClassifier(max_iter=500, random_state=seed).fit(X_tr, y_tr).score(X_te, y_te))
            run(n, "MLP, all n inputs (ours)", lambda: mlp_run(n, seed, *D))
            run(n, "Sparse polynomial LR (ours)", lambda: sparse_poly(*D))
        print(f"n={n} seed={seed} done in {time.time() - t0:.0f} s: " +
              ", ".join(f"{m.split(' (')[0]} {v[-1]:.3f}" for m, v in results[n].items()), flush=True)
save_json({"accuracy": {str(n): r for n, r in results.items()},
           "seconds": {str(n): t for n, t in seconds.items()}}, "model_comparison.json")

[track] 03_models_n20_s0: 23.6 s
n=20 seed=0 done in 24 s: Logistic regression 0.500, Quadratic LR 0.554, Poly-kernel LR, d=10, all bits 0.585, k-nearest neighbours 0.583, RBF-kernel SVM 0.569, Random forest 0.512, Gradient-boosted trees 0.564, MLP, all n inputs 0.935, Sparse polynomial LR 0.954


[track] 03_models_n20_s1: 82.0 s
n=20 seed=1 done in 82 s: Logistic regression 0.502, Quadratic LR 0.552, Poly-kernel LR, d=10, all bits 0.590, k-nearest neighbours 0.570, RBF-kernel SVM 0.580, Random forest 0.527, Gradient-boosted trees 0.585, MLP, all n inputs 0.932, Sparse polynomial LR 0.945


[track] 03_models_n20_s2: 59.8 s
n=20 seed=2 done in 60 s: Logistic regression 0.502, Quadratic LR 0.533, Poly-kernel LR, d=10, all bits 0.585, k-nearest neighbours 0.565, RBF-kernel SVM 0.584, Random forest 0.529, Gradient-boosted trees 0.545, MLP, all n inputs 0.937, Sparse polynomial LR 0.960


[track] 03_models_n20_s3: 74.0 s
n=20 seed=3 done in 74 s: Logistic regression 0.508, Quadratic LR 0.528, Poly-kernel LR, d=10, all bits 0.570, k-nearest neighbours 0.560, RBF-kernel SVM 0.565, Random forest 0.529, Gradient-boosted trees 0.559, MLP, all n inputs 0.941, Sparse polynomial LR 0.965


[track] 03_models_n20_s4: 98.6 s
n=20 seed=4 done in 99 s: Logistic regression 0.505, Quadratic LR 0.528, Poly-kernel LR, d=10, all bits 0.593, k-nearest neighbours 0.565, RBF-kernel SVM 0.586, Random forest 0.508, Gradient-boosted trees 0.571, MLP, all n inputs 0.948, Sparse polynomial LR 0.955


[track] 03_models_n30_s0: 93.3 s
n=30 seed=0 done in 93 s: Logistic regression 0.491, Quadratic LR 0.529, Poly-kernel LR, d=10, all bits 0.545, k-nearest neighbours 0.532, RBF-kernel SVM 0.552, Random forest 0.511, Gradient-boosted trees 0.515, MLP, all n inputs 0.955, Sparse polynomial LR 0.961


... (5 outputs left out)

[track] 03_models_n40_s1: 546.3 s
n=40 seed=1 done in 546 s: Logistic regression 0.497, Quadratic LR 0.530, Poly-kernel LR, d=10, all bits 0.536, k-nearest neighbours 0.528, RBF-kernel SVM 0.536, Random forest 0.531, Gradient-boosted trees 0.643, MLP, all n inputs 0.951, Sparse polynomial LR 0.956


[track] 03_models_n40_s2: 545.9 s
n=40 seed=2 done in 546 s: Logistic regression 0.505, Quadratic LR 0.523, Poly-kernel LR, d=10, all bits 0.539, k-nearest neighbours 0.535, RBF-kernel SVM 0.537, Random forest 0.518, Gradient-boosted trees 0.508, MLP, all n inputs 0.955, Sparse polynomial LR 0.959


[track] 03_models_n40_s3: 509.1 s
n=40 seed=3 done in 509 s: Logistic regression 0.499, Quadratic LR 0.535, Poly-kernel LR, d=10, all bits 0.540, k-nearest neighbours 0.527, RBF-kernel SVM 0.527, Random forest 0.515, Gradient-boosted trees 0.505, MLP, all n inputs 0.955, Sparse polynomial LR 0.960


[track] 03_models_n40_s4: 492.8 s
n=40 seed=4 done in 493 s: Logistic regression 0.500, Quadratic LR 0.534, Poly-kernel LR, d=10, all bits 0.538, k-nearest neighbours 0.528, RBF-kernel SVM 0.527, Random forest 0.514, Gradient-boosted trees 0.547, MLP, all n inputs 0.952, Sparse polynomial LR 0.959


## Table 1

In [4]:
models = list(results[NS[0]])
print(f"{'Model':42s}" + "".join(f"{'n=' + str(n):>16s}" for n in NS))
for m in models:
    print(f"{m:42s}" + "".join(f"{fmt(results[n][m]):>16s}" for n in NS))
print("\nLaTeX rows:")
for m in models:
    cells = []
    for n in NS:
        mu, h = mean_ci(results[n][m])
        cell = f"{100 * mu:.1f} $\\pm$ {100 * h:.1f}"
        cells.append(f"\\textbf{{{cell}}}" if mu >= 0.9 else cell)
    print(f"{m} & " + " & ".join(cells) + r" \\")

Model                                                 n=20            n=30            n=40
Logistic regression (linear)                    50.4 ± 0.4      50.3 ± 1.1      50.2 ± 0.6
Quadratic LR (mock paper)                       53.9 ± 1.6      52.9 ± 1.2      53.1 ± 0.6
Poly-kernel LR, d=10, all bits (mock)           58.5 ± 1.1      54.8 ± 1.1      54.0 ± 0.6
k-nearest neighbours (k tuned)                  56.9 ± 1.1      53.3 ± 0.8      53.0 ± 0.4
RBF-kernel SVM                                  57.7 ± 1.1      55.1 ± 0.7      53.2 ± 0.6
Random forest (300 trees)                       52.1 ± 1.3      51.8 ± 1.1      51.9 ± 0.9
Gradient-boosted trees                          56.5 ± 1.8      52.3 ± 1.6      54.2 ± 7.3
MLP, all n inputs (ours)                        93.8 ± 0.7      94.9 ± 0.8      95.3 ± 0.2
Sparse polynomial LR (ours)                     95.6 ± 0.9      96.0 ± 0.4      95.8 ± 0.2

LaTeX rows:
Logistic regression (linear) & 50.4 $\pm$ 0.4 & 50.3 $\pm$ 1.1 & 50.2 $\pm$ 0